# 06: Hyperparameter Tuning of the Risk MLP


In [1]:
import json
import os
from itertools import product

import numpy as np
import pandas as pd

In [2]:
# Run this notebook from the `ai/` directory.
HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "data")
IN_PATH = os.path.join(DATA_DIR, "pre_process_transaction_ds.csv")
TUNED_WEIGHTS = os.path.join(DATA_DIR, "tuned_model_weights.json")
BEST_PARAMS = os.path.join(DATA_DIR, "best_hyperparams.json")

In [3]:
FEATURE_NAMES = [
    "txVolume",            # tanh(log1p(avg value moved in ETH) / 3)
    "txFrequency",         # tanh(tx count / 50)
    "accountAge",          # tanh(active days / 365)
    "networkDegree",       # tanh(unique counterparties / 20)
    "timeRegularity",      # tanh(log1p(avg minutes between txs) / 6)
    "valueSentRatio",      # sent value / (sent + received)  -> 0..1, 0.5 neutral
    "inOutRatio",          # received txns / total txns      -> 0..1
    "degreeConcentration", # unique counterparties / tx count -> 0..1
    "valueVolatility",     # tanh(log1p(max value moved) / 3)
]

In [4]:
df = pd.read_csv(IN_PATH)
X = df[FEATURE_NAMES].to_numpy()
y = df["FLAG"].to_numpy()
print(f"Loaded {len(df):,} samples from {IN_PATH}")
print(f"Label balance: fraud={int(y.sum()):,} legit={int((1 - y).sum()):,}")

Loaded 9,841 samples from /home/ayush/Desktop/code/SecureTransac/ai/data/pre_process_transaction_ds.csv
Label balance: fraud=2,179 legit=7,662


In [5]:
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


def balanced_sample_weights(y):
    classes = np.unique(y)
    w = compute_class_weight("balanced", classes=classes, y=y)
    weights = np.ones(len(y), dtype=float)
    for c, cw in zip(classes, w):
        weights[y == c] = cw
    return weights

print(f"Train: {len(X_train)}  Test: {len(X_test)}")

Train: 7872  Test: 1969


In [6]:
GRID_SPACE = {
    "hidden_layer_sizes": [(12, 6), (16, 8), (32, 16), (64, 32), (24, 12), (32,)],
    "activation": ["relu", "tanh"],
    "alpha": [1e-4, 1e-3, 1e-2],
    "learning_rate_init": [1e-3, 5e-4, 1e-4],
}

# Notebook 03's hand-picked config (must be part of the search).
BASELINE = {
    "hidden_layer_sizes": (12, 6),
    "activation": "relu",
    "alpha": 1e-4,
    "learning_rate_init": 1e-3,
}

# Deterministic random sample of the cartesian grid (baseline always included).
keys = list(GRID_SPACE)
grid = list(product(*[GRID_SPACE[k] for k in keys]))
rng = np.random.default_rng(42)
N_RANDOM = 15
picked = rng.choice(len(grid), size=N_RANDOM, replace=False)
combos = [BASELINE]
for i in picked:
    combos.append(dict(zip(keys, grid[int(i)])))
print(f"Searching {len(combos)} configs (baseline + {N_RANDOM} random) over 3-fold CV")

Searching 16 configs (baseline + 15 random) over 3-fold CV


In [7]:
BASE_FIT = dict(
    solver="adam", max_iter=1500, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=10, random_state=42,
)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
folds = list(skf.split(X_train, y_train))

rows = []
for cfg in combos:
    fit_kwargs = {**BASE_FIT, **cfg}
    aucs, f1s = [], []
    for tr_idx, va_idx in folds:
        m = MLPClassifier(**fit_kwargs)
        m.fit(X_train[tr_idx], y_train[tr_idx],
              sample_weight=balanced_sample_weights(y_train[tr_idx]))
        p = m.predict_proba(X_train[va_idx])[:, 1]
        aucs.append(roc_auc_score(y_train[va_idx], p))
        f1s.append(f1_score(y_train[va_idx], (p >= 0.5).astype(int)))
    rows.append({**cfg, "val_auc": float(np.mean(aucs)), "val_f1": float(np.mean(f1s))})
    print(f"  {str(cfg['hidden_layer_sizes']):>10} {cfg['activation']:>5} "
          f"alpha={cfg['alpha']:.1e} lr={cfg['learning_rate_init']:.1e} "
          f"-> val AUC {np.mean(aucs):.4f} F1 {np.mean(f1s):.4f}", flush=True)

results = pd.DataFrame(rows).sort_values("val_auc", ascending=False).reset_index(drop=True)
results.head(10)

     (12, 6)  relu alpha=1.0e-04 lr=1.0e-03 -> val AUC 0.9482 F1 0.7359


     (12, 6)  relu alpha=1.0e-02 lr=1.0e-04 -> val AUC 0.8761 F1 0.6492


     (12, 6)  tanh alpha=1.0e-04 lr=1.0e-03 -> val AUC 0.9361 F1 0.7312


    (64, 32)  relu alpha=1.0e-02 lr=1.0e-04 -> val AUC 0.9149 F1 0.7200


    (24, 12)  tanh alpha=1.0e-04 lr=1.0e-03 -> val AUC 0.9416 F1 0.7247


...

    (64, 32)  relu alpha=1.0e-04 lr=1.0e-03 -> val AUC 0.9657 F1 0.7751


    (24, 12)  relu alpha=1.0e-04 lr=5.0e-04 -> val AUC 0.9548 F1 0.7505


    (64, 32)  tanh alpha=1.0e-02 lr=5.0e-04 -> val AUC 0.9427 F1 0.7304


       (32,)  tanh alpha=1.0e-03 lr=1.0e-03 -> val AUC 0.9420 F1 0.7300


,hidden_layer_sizes,activation,alpha,learning_rate_init,val_auc,val_f1
0,"(64, 32)",relu,0.0001,0.0010,0.965737,0.775074
1,"(24, 12)",relu,0.0001,0.0005,0.954807,0.750509
2,"(32, 16)",relu,0.0100,0.0010,0.952616,0.745755
3,"(12, 6)",relu,0.0001,0.0010,0.948194,0.735885
4,"(32,)",relu,0.0100,0.0005,0.944255,0.729054
5,"(24, 12)",tanh,0.0010,0.0005,0.943079,0.735268
6,"(64, 32)",tanh,0.0100,0.0005,0.942696,0.730436
7,"(32,)",tanh,0.0001,0.0010,0.942025,0.729999
8,"(32,)",tanh,0.0010,0.0010,0.942025,0.729999
9,"(24, 12)",tanh,0.0001,0.0010,0.941625,0.724654


In [8]:
def fit_and_eval(cfg):
    m = MLPClassifier(**{**BASE_FIT, **cfg})
    m.fit(X_train, y_train, sample_weight=balanced_sample_weights(y_train))
    p = m.predict_proba(X_test)[:, 1]
    return m, {
        "auc": float(roc_auc_score(y_test, p)),
        "f1": float(f1_score(y_test, (p >= 0.5).astype(int))),
    }

PARAM_KEYS = ("hidden_layer_sizes", "activation", "alpha", "learning_rate_init")
best_cfg = {k: v for k, v in results.iloc[0].items() if k in PARAM_KEYS}
best_cfg["hidden_layer_sizes"] = tuple(best_cfg["hidden_layer_sizes"])

best_model, best_metrics = fit_and_eval(best_cfg)
_, base_metrics = fit_and_eval(BASELINE)

comparison = pd.DataFrame([
    {"model": "MLP baseline (12, 6)", **base_metrics},
    {"model": f"MLP tuned {best_cfg['hidden_layer_sizes']}", **best_metrics},
])
comparison

,model,auc,f1
0,"MLP baseline (12, 6)",0.952803,0.761811
1,"MLP tuned (64, 32)",0.966959,0.813094


In [9]:
improved = best_metrics["auc"] >= base_metrics["auc"]
print(f"Baseline test AUC: {base_metrics['auc']:.4f}")
print(f"Tuned    test AUC: {best_metrics['auc']:.4f}  -> {'IMPROVED' if improved else 'no gain'}")

with open(BEST_PARAMS, "w") as f:
    json.dump(best_cfg, f, indent=2)
print(f"Wrote {BEST_PARAMS}")

if improved:
    last = best_model.coefs_[-1]
    assert last.shape[1] == 1, f"Unexpected binary output shape {last.shape}"
    weights = {"layers": []}
    for i in range(len(best_model.coefs_)):
        weights["layers"].append({
            "weights": best_model.coefs_[i].tolist(),
            "biases": best_model.intercepts_[i].tolist(),
            "activation": "relu" if i < len(best_model.coefs_) - 1 else "sigmoid",
        })
    with open(TUNED_WEIGHTS, "w") as f:
        json.dump(weights, f)
    print(f"Exported {TUNED_WEIGHTS} ({os.path.getsize(TUNED_WEIGHTS):,} bytes, "
          f"{len(weights['layers'])} layers)")
else:
    print("No improvement; tuned weights NOT exported.")

Baseline test AUC: 0.9528
Tuned    test AUC: 0.9670  -> IMPROVED
Wrote /home/ayush/Desktop/code/SecureTransac/ai/data/best_hyperparams.json
Exported /home/ayush/Desktop/code/SecureTransac/ai/data/tuned_model_weights.json (60,341 bytes, 3 layers)


In [10]:
# Confirm on the full dataset with 5-fold CV (same protocol as notebook 03).
cv_results = {}
for name, cfg in [("baseline", BASELINE), ("tuned", best_cfg)]:
    aucs = []
    for tr_idx, va_idx in skf.split(X, y):
        m = MLPClassifier(**{**BASE_FIT, **cfg})
        m.fit(X[tr_idx], y[tr_idx], sample_weight=balanced_sample_weights(y[tr_idx]))
        aucs.append(roc_auc_score(y[va_idx], m.predict_proba(X[va_idx])[:, 1]))
    cv_results[name] = (float(np.mean(aucs)), float(np.std(aucs)))
    print(f"{name:>9} 5-fold CV AUC: {np.mean(aucs):.4f} (+/- {np.std(aucs):.4f})")

best_params_summary = {
    "best": best_cfg,
    "test_auc": best_metrics["auc"],
    "baseline_test_auc": base_metrics["auc"],
    "cv_auc_mean": cv_results["tuned"][0],
    "cv_auc_std": cv_results["tuned"][1],
    "baseline_cv_auc_mean": cv_results["baseline"][0],
}
with open(BEST_PARAMS, "w") as f:
    json.dump(best_params_summary, f, indent=2)
print(f"Updated {BEST_PARAMS}")

 baseline 5-fold CV AUC: 0.9530 (+/- 0.0030)


    tuned 5-fold CV AUC: 0.9590 (+/- 0.0108)
Updated /home/ayush/Desktop/code/SecureTransac/ai/data/best_hyperparams.json
